#### Importing Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from sbi.utils import BoxUniform
from torch.distributions import LogNormal, Independent
from joblib import Parallel, delayed
from sbi.analysis import pairplot
from sbi.inference import NPE
from sbi.analysis import plot_summary
import matplotlib.pyplot as plt
import camb
import healpy as hp

_ = torch.manual_seed(42)

_ = np.random.seed(0)


#### Defining Fiducial Params, generating compressor

In [ ]:
param_dict={
    "H0":67.5, "ombh2":0.022, "omch2":0.122, "mnu":0.06, "omk":0, "tau":0.06, "As":2e-9, "ns":0.965, "halofit_version":'mead', "lmax":3000
} # using placnk guess to be close to true values

In [ ]:
# which parameters to vary for compression. The value is a fractional step size for numerical derivatives
derivatives = {"H0":0.001, "ombh2":0.005, "omch2":0.005,"As":0.005, "ns":0.005} 

In [ ]:
# Note I convert As to As 10^10 in the derivatives to give more reasonable numbers.

def generateMock(cl,nls=None,lmax=3000,beam_fwhm=0):
    """
      could put camb call into here too.
      nls a list of noise spectra for each split [nl,nl...].
      Beams are a second observational effect. 1-7 is a reasonable value. Could also be different per split... """
    cmb_alm = hp.synalm(cl,lmax=lmax) # generates a 2d spherical sky map from 1d spectrum
    if beam_fwhm is not None and beam_fwhm!=0:
        beam = hp.gauss_beam(beam_fwhm/60/180*np.pi,lmax=lmax) # simulates telescope blur with a gaussian blur and appliies to sky map
    else:
        beam=np.ones(3000)
        #cmb_alm = hp.almxfl(
    if nls is not None:
        obs_alms = []
        for nl in nls:
            noise = hp.synalm(nl,lmax=lmax) # telescope's electronic noise
            obs_alms.append(hp.almxfl(cmb_alm,beam)+noise) # combine blur plus noise
        return np.array(obs_alms)
    else:
        return cmb_alm





def getCompression(param_dict,derivatives,beam_fwhm=None,noise_cl=None):
    """ 
    A simple implementation of the moped algorithm. This assumes no dependence of cov mat on parameters (e.g. https://arxiv.org/pdf/1204.4724)
    Compute derivatives numerically
    Compute the cov mat based on a gaussian formalua. 
    Compressor is then \nabla \mu C^{-1} (d-\mu). 
    Note: Match the beam and noise to the splits above.
    This is a close to optimal compression around the fiducial cosmology. Maybe use an SBI prior is some region around here. Planck +3 sigma?
    """
    
    lmax=param_dict['lmax']
    def getSpectrum(params): # similar function to generateMock, adds some noise
        pars = camb.set_params(**params)
        results = camb.get_results(pars)
        powers = results.get_cmb_power_spectra(pars, CMB_unit='muK',raw_cl=True)['total'][2:3001,0]
        if noise_cl is None:
            noise=0
        else:
            noise=noise_cl[2:3001]
        if beam_fwhm is not None:
            beam = hp.gauss_beam(beam_fwhm/60/180*np.pi,lmax=lmax)[2:]
        else:
            beam = np.ones(3001)[2:]
    
        return powers*beam**2+noise # Add in the effect of beam and noise if you want them
    fiducial = getSpectrum(param_dict) # run simulation once to obtain "expected" observation
    cov_mat = 2./(2*np.arange(2,3001)+1)*fiducial**2 # Gaussian cov mat.: error dominated by cosmic variance (have one universe to look at, 2/(2l+1)*C_ell^2)
    derivs = []
    for deriv_name in derivatives.keys():
        step = param_dict[deriv_name]*derivatives[deriv_name]
        deriv_params = dict(param_dict)
        deriv_params[deriv_name]+=step
        up = getSpectrum(deriv_params)
        deriv_params = dict(param_dict)
        deriv_params[deriv_name]-=step
        down = getSpectrum(deriv_params)
        if deriv_name=='As': step*=1e10 # Different "units" for this to adjust ranges of scales (measuring As 10^10 instead of As)
        derivs.append((up-down)/(2*step)) # we calculate gradients by finding difference beteeen slight nudge up and down
        print(deriv_name)
    derivs = np.array(derivs)
    def compressor(data):
        data = data[2:]# Remove first two elements, monopole and dipoel
        return  np.dot(derivs,(data-fiducial)/cov_mat) 
    return fiducial,cov_mat,derivs,compressor

In [ ]:
beam_fwhm = 5.
nl = np.ones(5000)*(20/60/180*60)**2 # White noise spectrum
alms = generateMock(powers[:,0],nls=[nl,1.3*nl],beam_fwhm=beam_fwhm)

In [ ]:
test = getCompression(param_dict,derivatives,beam_fwhm=beam_fwhm,noise_cl=nl)

#### Defining Functions

In [ ]:

# Define Lotka Volterra Simulator
def lotka_volterra(y, alpha, beta, delta, gamma):
    prey, predator = y
    dprey_dt = alpha * prey - beta * prey * predator
    dpredator_dt = delta * prey * predator - gamma * predator
    return np.asarray([dprey_dt, dpredator_dt])

def simulate_total(parameters, t_span):
    alpha = parameters[0]
    beta = parameters[1]
    delta = parameters[2]
    gamma = parameters[3]

    y0 = np.asarray([40.0, 9.0])  # Initial populations
    #t_span = 400  # Total simulation time
    dt = 0.1  # Time step

    timesteps = int(2*t_span / dt)
    y = np.zeros((timesteps, 2))
    y[0] = y0

    for i in range(1, timesteps):
        y[i] = y[i-1] + lotka_volterra(y[i-1], alpha, beta, delta, gamma) * dt

    return y

def choose_splitter(nparray, split_method):
    if split_method == 'even_odd':
        return split_by_even_odd(nparray)
    elif split_method == 'first_second':
        return split_in_two(nparray)
    else:
        raise ValueError(f"Unknown split type: {split_method}")
    

# split into two halves
def split_in_two(nparray):
    n = len(nparray)
    mid = n//2
    res1 = nparray[:mid]
    res2 = nparray[mid:]
    return res1, res2

# split by even/odd indices
def split_by_even_odd(nparray):
    even = nparray[::2]
    odd = nparray[1::2]
    m = min(len(even), len(odd))
    even, odd = even[:m], odd[:m]
    return even, odd


# Define how populations evolve
def simulate(parameters, observation, t_span):
    alpha = parameters[0]
    beta = parameters[1]
    delta = parameters[2]
    gamma = parameters[3]


    y0 = np.asarray([observation[0, 0], observation[0, 1]])  # Initial populations
    #t_span = 200  # Total simulation time
    dt = 0.1  # Time step

    timesteps = int(t_span / dt)
    y = np.zeros((timesteps, 2))
    y[0] = y0

    for i in range(1, timesteps):
        y[i] = y[i-1] + lotka_volterra(y[i-1], alpha, beta, delta, gamma) * dt

    return y

# Extract observed data
def extract_data(url):
    df = pd.read_csv(url, delim_whitespace=True, header=None, index_col=0)
    df.index.name = 'Year'
    df.columns = ['Hare', 'Lynx']
    time_vec = df.index.values
    observation = df[['Hare', 'Lynx']].values
    print(observation.shape)
    n_obs = observation.shape[0]
    sigma_hare = 0.2 * np.std(observation[:, 0])   # 20% of hare std
    sigma_lynx = 0.2 * np.std(observation[:, 1])   # 20% of lynx std
    return time_vec, observation, n_obs, sigma_hare, sigma_lynx

# Plot figures for observed lynx hare populations
def plot_observed_data(time_vec, observation):
    fig, ax = plt.subplots(1, 1, figsize=(6, 3))
    _ = ax.plot(time_vec, observation)
    _ = ax.legend(["Prey", "Predator"])
    _ = ax.set_xlabel("Time")
    _ = ax.set_ylabel("Population")

# Use mean and max as summary statistics but add noise: intended for our actual observation
def add_noise_and_plot(simulation_result, distn, sigma_hare, sigma_lynx, time):

    if distn == 'none':
        pass
    elif distn == 'gaussian':
        noise = np.random.normal(
        loc=0.0,
        scale=np.array([sigma_hare, sigma_lynx]),  # one sigma per column
        size=simulation_result.shape)
        simulation_result = simulation_result + noise
    elif distn == 'poisson':
        lam = np.clip(simulation_result, 1e-6, None)
        simulation_result = np.random.poisson(lam)
    else:
        raise ValueError(f"Unknown distribution type: {distn}")

    plot_observed_data(time,simulation_result)

    
    return simulation_result

# Use mean and max as summary statistics
def summarize_simulation(simulation_result):

    prey_population = simulation_result[:, 0]
    predator_population = simulation_result[:, 1]
    
    summary = [
        np.max(prey_population).item(),
        np.max(predator_population).item(),
        np.mean(prey_population).item(),
        np.mean(predator_population).item()
    ]
    return np.asarray(summary)

# pick out n_obs points to ensure arrays are same size
def downsample_to_n_obs(sim_result, n_obs):
    # pick out n_obs points to ensure arrays are same size
    T = sim_result.shape[0]
    idx = np.linspace(0, T - 1, n_obs).astype(int)
    return sim_result[idx]

# returns data that is same size as observed, adding no noise
def simulate_match_data(parameters, n_obs,observation, t_span):
    # returns data that is same size as observed, adding no noise
    full_traj = simulate(parameters,observation,t_span)            # (2000, 2)
    return downsample_to_n_obs(full_traj, n_obs)  # (n_obs, 2)

# Returns simulations with gaussian noise added
def simulate_gaussian(parameters, n_obs, sigma_hare, sigma_lynx, observation, t_span):
    latent = simulate_match_data(parameters, n_obs,observation, t_span)
    noise = np.random.normal(
        loc=0.0,
        scale=np.array([sigma_hare, sigma_lynx]),  # one sigma per column
        size=latent.shape
    )
    # Column 0 ~ N(0, sigma_hare^2), column 1 ~ N(0, sigma_lynx^2)
    noisy = latent + noise
    return noisy

# Return simulations with poissoin noise added
def simulate_poisson(parameters, n_obs, observation, t_span):
    latent = simulate_match_data(parameters, n_obs, observation, t_span)
    lam = np.clip(latent, 1e-6, None)
    noisy_counts = np.random.poisson(lam)
    return noisy_counts

# Wrapper function to choose noise
def simulator_distribution(distn, parameters, n_obs, sigma_hare, sigma_lynx, observation, t_span):
    if distn == 'none':
        result = simulate_match_data(parameters, n_obs, observation,t_span)
    elif distn == 'gaussian':
        result = simulate_gaussian(parameters, n_obs, sigma_hare, sigma_lynx, observation, t_span)
    elif distn == 'poisson':
        result = simulate_poisson(parameters, n_obs, observation, t_span)
    else:
        raise ValueError(f"Unknown distribution type: {distn}")
    return result

# Define a uniform prior
def define_uniform_prior():
    lower_bound = torch.as_tensor([0.05, 0.01, 0.005, 0.005])
    upper_bound = torch.as_tensor([0.15, 0.03, 0.03, 0.15])
    prior = BoxUniform(low=lower_bound, high=upper_bound)
    return prior

# Define a log-normal prior
def define_logn_prior():
    log_means = torch.log(torch.tensor([
        0.13,   # alpha - 1.6   old: 0.1
        0.015,   # beta - 3.2   old: 0.02
        0.008,  # delta - 0.6    old: 0.015
        0.12    # gamma - 0.56     old: 0.08
    ]))

    log_stds = torch.tensor([0.3, 0.3, 0.3, 0.3]) # old was just 0.3

    # Build prior
    base_dist = LogNormal(loc=log_means, scale=log_stds)
    prior_lognormal = Independent(base_dist, 1)
    return prior_lognormal


# Wrapper Function to choose prior distn
def choose_prior_and_generate_theta(distn):
    if distn == 'uniform':
        prior = define_uniform_prior()
    elif distn == 'logn':
        prior = define_logn_prior()
    else:
        raise ValueError(f"Unknown distribution type: {distn}")
    
    theta = prior.sample((10_000,))
    return prior, theta

# simulator function
def parallel_simulate(theta,distn, n_obs, sigma_hare, sigma_lynx, observation, t_span):
    # Our simulator uses numpy, but prior samples are in PyTorch.
    theta_np = theta.numpy()

    num_workers = 8
    simulation_outputs = Parallel(n_jobs=num_workers)(
        delayed(simulator_distribution)(distn,batch,n_obs, sigma_hare, sigma_lynx, observation, t_span)
        for batch in theta_np
    )
    return np.asarray(simulation_outputs)

# generate simulations
def generate_x(theta,distn, n_obs,sigma_hare,sigma_lynx, observation, t_span):
    simulation_outputs = parallel_simulate(theta,distn, n_obs,sigma_hare, sigma_lynx, observation, t_span)
    x = torch.as_tensor(np.asarray([summarize_simulation(sim) for sim in simulation_outputs]), dtype=torch.float32)
    return x

# plot to check simulated data covers observed data
def plot_checker(x, x_obs):
        _ = pairplot(
        samples=x,
        points=x_obs[None, :],  # `points` needs a batch dimension.
        limits=[[0, 200], [0, 150], [0, 25], [0, 50]],
        figsize=(4, 4),
    )


In [ ]:
# Generate net, stopping after max epochs, choosing or not choosing to 
def train_net_generate_samples(x,theta,x_obs, prior, verbose, max_epoch,true_val):
    
    inference = NPE(prior= prior, density_estimator="nsf")
    posterior_net = inference.append_simulations(theta, x).train(max_num_epochs=max_epoch, stop_after_epochs=30)
    posterior_direct = inference.build_posterior(density_estimator=posterior_net,sample_with="direct")
    posterior_mcmc = inference.build_posterior(density_estimator=posterior_net,sample_with="mcmc")
    samples = posterior_mcmc.sample((1_000,), x=x_obs)

    if verbose == True:
        _ = plot_summary(
        inference,
        tags=["training_loss", "validation_loss"],
        figsize=(10, 2),
        )

        print(posterior_mcmc)
        print("Observation: ", x_obs)

        _ = pairplot(
            samples,
            points = true_val[None,:],
            limits=[[0.05, 0.2], [0.01, 0.05], [0.001, 0.02], [0.005, 0.2]],
            ticks=[[0.05, 0.2], [0.01, 0.05], [0.001, 0.02], [0.005, 0.2]],
            figsize=(5, 5),
            labels=[r"$\alpha$", r"$\beta$", r"$\delta$", r"$\gamma$"]
            )
        
    return samples, posterior_mcmc, posterior_direct, inference

In [ ]:
t_span = 200
time_vec = np.arange(0, 2*t_span, 0.1)
true_parameters = np.asarray([0.1, 0.02, 0.01, 0.1])
split_type =  "even_odd" #"first_second"

observation = simulate_total(true_parameters,t_span)
plot_observed_data(time_vec, observation) # total true obs

time1, time2 = choose_splitter(time_vec,split_type)
obs1, obs2 = choose_splitter(observation,split_type)

plot_observed_data(time1, obs1)
plot_observed_data(time2, obs2) # THIS IS OBSERVATIOSN WITHOUT NOISE ADDED

In [ ]:
sigma_hare1 = 0.2 * np.std(obs1[:, 0])   # 20% of hare std
sigma_lynx1 = 0.2 * np.std(obs1[:, 1])   # 20% of lynx std
sigma_hare2 = 0.2 * np.std(obs2[:, 0])   # 20% of hare std
sigma_lynx2 = 0.2 * np.std(obs2[:, 1])   # 20% of lynx std
n_obs1 = len(obs1)
n_obs2 = len(obs2)

prior_name = 'logn' #prior =  'uniform' 
noise1 =   'poisson' #'gaussian' 'none'
noise2 = 'gaussian'
max_epoch = 1000

prior1,theta1 = choose_prior_and_generate_theta(prior_name)

x1 = generate_x(theta1,noise1, n_obs1, sigma_hare1, sigma_lynx1, obs1, t_span)

noisy_obs1= add_noise_and_plot(obs1,noise1,sigma_hare1,sigma_lynx1,time1)
x_obs1 = summarize_simulation(noisy_obs1)
plot_checker(x1,x_obs1)


In [ ]:
prior2,theta2 = choose_prior_and_generate_theta(prior_name)
x2 = generate_x(theta2,noise2, n_obs2, sigma_hare2, sigma_lynx2, obs2, t_span)

noisy_obs2= add_noise_and_plot(obs2,noise2,sigma_hare2,sigma_lynx2,time2)
x_obs2 = summarize_simulation(noisy_obs2)
plot_checker(x2,x_obs2)

In [ ]:
samples1, posterior1_mcmc, posterior1_direct, inference1 = train_net_generate_samples(x1,theta1,x_obs1, prior1, verbose = True, max_epoch=1000, true_val=true_parameters)

In [ ]:
_ = pairplot(
            samples1,
            points = true_parameters[None,:],
            limits=[[0.05, 0.2], [0.01, 0.05], [0.001, 0.02], [0.005, 0.2]],
            ticks=[[0.05, 0.2], [0.01, 0.05], [0.001, 0.02], [0.005, 0.2]],
            figsize=(5, 5),
            labels=[r"$\alpha$", r"$\beta$", r"$\delta$", r"$\gamma$"]
            )

In [ ]:
samples2, posterior2_mcmc, posterior2_direct, inference2 = train_net_generate_samples(x2,theta2,x_obs2, prior2, verbose = True, max_epoch=1000, true_val=true_parameters)

In [ ]:
_ = pairplot(
            samples2,
            points = true_parameters[None,:],
            limits=[[0.05, 0.11], [0.01, 0.025], [0.001, 0.013], [0.01, 0.11]],
            ticks=[[0.05, 0.11], [0.01, 0.025], [0.001, 0.013], [0.01, 0.11]],
            figsize=(5, 5),
            labels=[r"$\alpha$", r"$\beta$", r"$\delta$", r"$\gamma$"]
            )

In [ ]:
results = {
        "samples1": samples1,          # posterior samples (1000, 4)
        "samples2": samples2,          # posterior samples (1000, 4)
        "posterior1_mcmc": posterior1_mcmc,   # sbi posterior object (if you want to reuse it)
        "posterior1_direct": posterior1_direct,   # sbi posterior object (if you want to reuse it)
        "posterior2_mcmc": posterior2_mcmc,   # sbi posterior object (if you want to reuse it)
        "posterior2_direct": posterior2_direct,   # sbi posterior object (if you want to reuse it)
        "inference1": inference1,      # NPE object
        "inference2": inference2,      # NPE object
        "prior1": prior1,
        "prior2": prior2,
        "theta1": theta1,              # simulated θ used for training (10000, 4)
        "theta2": theta2,              # simulated θ used for training (10000, 4)
        "x1": x1,                      # simulated summaries (10000, 4)
        "x2": x2,                      # simulated summaries (10000, 4)
        "x_obs1": x_obs1,              # observed summary (4,)
        "x_obs2": x_obs2,              # observed summary (4,)
        "time_vec1": time1,        # years
        "time_vec2": time2,        # years
        "obs1": obs1,               # raw data (n_obs, 2)
        "obs2": obs2,               # raw data (n_obs, 2)
        "prior_name": prior_name,    # 'uniform' or 'logn'
        "noise1": noise1,        # 'none' / 'gaussian' / 'poisson'
        "noise2": noise2,        # 'none' / 'gaussian' / 'poisson'
        "n_obs1": n_obs1,
        "n_obs2": n_obs2,
        "sigma_hare1": sigma_hare1,
        "sigma_lynx1": sigma_lynx1,
        "sigma_hare2": sigma_hare2,
        "sigma_lynx2": sigma_lynx2,
        "split_type": split_type,
        "t_span": t_span,
        "true_parameters": true_parameters
    }


fname = f"CALIBRATION TESTER_split_{split_type}_results_{prior_name}_{noise1}_1000EPOCH.pt"
torch.save(results, fname)